# Práctica 1 - Pipeline alternativo de Machine Learning

En esta práctica se construye un pipeline alternativo completo de Machine Learning para predecir el impago de préstamos. Para ello, se implementa una nueva clase de preprocesamiento y una nueva clase de filtrado, siguiendo el patrón `fit/transform` para evitar data leakage. Posteriormente, se entrenan tres modelos de distintas familias: un ensemble de árboles, una máquina de soporte vectorial y una red neuronal. Finalmente, se comparan sus resultados con el modelo base visto en clase.

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve, auc

from src.preprocessing.practica1_preprocessing import Practica1Preprocess
from src.filtering.practica1_filtering import Practica1Filtering

In [75]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="During the encoding, NaN values were introduced*"
)

## 1. Preprocesamiento de los datos

En primer lugar, se aplica la clase `Practica1Preprocess`, que utiliza el fichero `variables_withExperts.xlsx` para incluir también las variables procedentes de evaluaciones de expertos, como `grade`, `sub_grade`, `int_rate`, `installment` y las variables relacionadas con FICO.

Además, se emplean técnicas alternativas a las de la clase base: imputación de variables categóricas mediante una categoría constante para valores desconocidos, codificación ordinal para variables con orden natural, codificación por frecuencia para el resto de categóricas, escalado robusto de variables numéricas y creación de nuevas features financieras como `fico_mean`, `installment_income_ratio` y `loan_income_ratio`.

In [76]:
preprocessor = Practica1Preprocess(
    var_to_process="data/variables_withExperts.xlsx",
    target="loan_status"
)

preprocessor.fit("data/df_train_small.csv")

X_train_pre, y_train = preprocessor.transform("data/df_train_small.csv")
X_test_pre, y_test = preprocessor.transform("data/df_test_small.csv")

preprocessor.print_summary()

print("Shape X_train después de preprocesamiento:", X_train_pre.shape)
print("Shape X_test después de preprocesamiento:", X_test_pre.shape)
print("Shape y_train:", y_train.shape)
print("Shape y_test:", y_test.shape)

RESUMEN DEL PREPROCESAMIENTO PRACTICA 1
Variables predictoras usadas: 102
Variables numéricas originales: 92
Variables categóricas originales: 15
Variables ordinales: ['grade', 'sub_grade']
Variables con frequency encoding: 13
Variables numéricas finales escaladas: 107
Shape X_train después de preprocesamiento: (80000, 107)
Shape X_test después de preprocesamiento: (20000, 107)
Shape y_train: (80000,)
Shape y_test: (20000,)


## 2. Filtrado de variables

Una vez preprocesados los datos, se aplica la clase `Practica1Filtering`. En este caso, se ha optado por un filtrado alternativo basado en dos pasos: eliminación de variables con varianza muy baja y selección de las variables más relevantes mediante información mutua. Este proceso permite reducir la dimensionalidad y conservar las características con mayor poder predictivo.

In [77]:
print("NaN en X_train_pre:", X_train_pre.isna().sum().sum())
print("NaN en X_test_pre:", X_test_pre.isna().sum().sum())

NaN en X_train_pre: 0
NaN en X_test_pre: 0


In [78]:
filtering = Practica1Filtering(variance_threshold=0.0, k_best=80)

filtering.fit(X_train_pre, y_train)

X_train_filt = filtering.transform(X_train_pre)
X_test_filt = filtering.transform(X_test_pre)

filtering.print_summary()

print("Shape X_train después de filtrado:", X_train_filt.shape)
print("Shape X_test después de filtrado:", X_test_filt.shape)

RESUMEN DEL FILTRADO PRACTICA 1
Features iniciales:                    107
Tras filtro de varianza:              107
Features seleccionadas finales:       80
Shape X_train después de filtrado: (80000, 80)
Shape X_test después de filtrado: (20000, 80)


## 3. Entrenamiento de modelos

Una vez finalizado el preprocesamiento y el filtrado, se entrenan tres modelos de aprendizaje automático pertenecientes a familias distintas. En concreto, se utilizará un ensemble de árboles de decisión, una máquina de soporte vectorial y una red neuronal. El objetivo es comparar su rendimiento en la predicción de impago sobre el conjunto de test.

### 3.1 Ensemble de árboles

Como modelo de tipo ensemble se ha elegido un `RandomForestClassifier`, ya que suele ofrecer un buen rendimiento en problemas tabulares, permite capturar relaciones no lineales entre variables y es relativamente robusto frente a ruido y outliers. Además, se ha configurado con `class_weight='balanced'` para tener en cuenta el desbalanceo entre clases.

In [79]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    class_weight={0: 1, 1: 3},
    random_state=42
)

hgb_model.fit(X_train_filt, y_train.values.ravel());

### 3.2 Máquina de soporte vectorial

Como representante de las máquinas de soporte vectorial se emplea un modelo `SVC` con kernel radial (`rbf`). Este tipo de modelo puede capturar fronteras de decisión no lineales y resulta útil como comparación frente a los modelos basados en árboles. Se activa la opción `probability=True` para poder calcular posteriormente la métrica PR-AUC.

In [80]:
# Para que no sea excesivamente lento, se toma una muestra del train
X_train_svm = X_train_filt.sample(n=10000, random_state=42)
y_train_svm = y_train.loc[X_train_svm.index]

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_svm, y_train_svm.values.ravel());

### 3.3 Red neuronal

Como tercer modelo se entrena una red neuronal multicapa mediante `MLPClassifier`. Este modelo permite aproximar relaciones complejas entre variables y constituye una tercera familia claramente distinta a las anteriores. Se ha escogido una arquitectura sencilla y un número moderado de iteraciones para mantener un equilibrio entre rendimiento y tiempo de entrenamiento.

In [81]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=400,
    early_stopping=True,
    random_state=42
)

mlp_model.fit(X_train_filt, y_train.values.ravel());

## 4. Evaluación de modelos

Para comparar los modelos entrenados, se calculan las métricas solicitadas en el conjunto de test: `accuracy`, `precision`, `recall` y `PR-AUC`. Dado que el problema presenta clases desbalanceadas, la métrica PR-AUC resulta especialmente relevante, ya que se centra en la capacidad del modelo para detectar correctamente la clase minoritaria, es decir, los casos de impago.

In [82]:
def evaluate_model(model, X_test, y_test, model_name):
    y_true = y_test.values.ravel()
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_vals, precision_vals)

    results = {
        "Modelo": model_name,
        "Accuracy": accuracy,
        "Precision (impago)": precision,
        "Recall (impago)": recall,
        "PR-AUC": pr_auc
    }

    return results

In [83]:
results_hgb = evaluate_model(hgb_model, X_test_filt, y_test, "HistGradientBoosting")
results_svm = evaluate_model(svm_model, X_test_filt, y_test, "SVM")
results_mlp = evaluate_model(mlp_model, X_test_filt, y_test, "MLP")

In [84]:
# Cálculo del PR-AUC del modelo base visto en clase
# El modelo base usa el FICO score normalizado como aproximación simple al riesgo.

df_base = pd.read_csv("data/df_test_small.csv")

df_fico = (
    df_base.loc[:, ["fico_range_low", "fico_range_high", "loan_status"]]
    .assign(prob_low=lambda x: (x.fico_range_low - 300) / (850 - 300))
    .assign(prob_high=lambda x: (x.fico_range_high - 300) / (850 - 300))
    .assign(prob_paid=lambda x: (x.prob_low + x.prob_high) / 2)
)

y_base_default = (df_fico["loan_status"] != "Fully Paid").astype(int)

# Como prob_paid mide probabilidad aproximada de pagar,
# para la clase positiva de impago usamos 1 - prob_paid.
prob_default = 1 - df_fico["prob_paid"]

precision_vals, recall_vals, _ = precision_recall_curve(
    y_base_default,
    prob_default
)

base_pr_auc = auc(recall_vals, precision_vals)

base_results = {
    "Modelo": "Modelo base",
    "Accuracy": 0.72,
    "Precision (impago)": 0.26,
    "Recall (impago)": 0.24,
    "PR-AUC": base_pr_auc
}

results_df = pd.DataFrame([
    base_results,
    results_hgb,
    results_svm,
    results_mlp
])

results_df

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.7200,0.260000,0.240000,0.291412
1,HistGradientBoosting,0.7147,0.359711,0.548161,0.389932
2,SVM,0.6931,0.228919,0.226170,0.230055
3,MLP,0.7973,0.462351,0.087566,0.307711


In [85]:
results_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision (impago)": "{:.3f}",
    "Recall (impago)": "{:.3f}",
    "PR-AUC": "{:.3f}"
})

,Modelo,Accuracy,Precision (impago),Recall (impago),PR-AUC
0,Modelo base,0.720,0.260,0.240,0.291
1,HistGradientBoosting,0.715,0.360,0.548,0.390
2,SVM,0.693,0.229,0.226,0.230
3,MLP,0.797,0.462,0.088,0.308


## 5. Comparación con el modelo base

La tabla de resultados muestra diferencias claras entre los modelos entrenados y el modelo base de referencia. Este baseline obtenía aproximadamente un 72% de accuracy, un 26% de precision en la clase de impago y un 24% de recall. Además, se ha calculado su PR-AUC usando el FICO score normalizado como aproximación a la probabilidad de pago y transformándolo a probabilidad de impago mediante `1 - prob_paid`.

El modelo que mejor detecta los casos de impago es el **HistGradientBoosting**, ya que alcanza un recall de 0.666, muy superior al 0.240 del modelo base. Además, también mejora la precision, pasando de 0.260 a 0.324, y obtiene una PR-AUC de 0.387, lo que indica un mejor rendimiento global sobre la clase minoritaria. Su principal inconveniente es que la accuracy baja hasta 0.656, lo que sugiere que detecta más impagos a costa de cometer más errores globales.

El modelo **SVM** ofrece un rendimiento inferior al baseline en casi todas las métricas relevantes. Su accuracy es de 0.693, su precision de 0.229 y su recall de 0.226, valores que no mejoran los del modelo base. Además, su PR-AUC es de 0.230, lo que confirma que su capacidad para discriminar los casos de impago es bastante limitada en este problema.

Por su parte, la **red neuronal MLP** obtiene la mejor accuracy de todos los modelos, con un valor de 0.797, y también mejora la precision respecto al baseline, alcanzando 0.462. Sin embargo, su recall cae hasta 0.088, lo que significa que apenas detecta una pequeña parte de los impagos reales. Por tanto, aunque acierta mucho en términos globales, no resulta especialmente útil si el objetivo principal es identificar clientes con riesgo de impago.

En conjunto, estos resultados muestran que el mejor modelo depende de la métrica que se considere prioritaria. Si el objetivo es maximizar la detección de impagos, el Random Forest es claramente la mejor opción. Si se prioriza la accuracy global, el MLP obtiene el mejor resultado, aunque a costa de perder gran parte de la capacidad de detección de la clase minoritaria.

## 6. Conclusiones

En esta práctica se ha construido un pipeline alternativo de Machine Learning para la predicción de impago, utilizando una estrategia de preprocesamiento, filtrado y modelado distinta a la empleada en clase. En particular, se han incorporado variables de expertos, se han aplicado métodos alternativos de imputación, codificación y escalado, y se han generado nuevas variables derivadas con significado financiero.

Tras el preprocesamiento y el filtrado, se obtuvo un conjunto final de 80 variables, con el que se entrenaron tres modelos de familias distintas: un Random Forest, una máquina de soporte vectorial y una red neuronal MLP. Esta comparación ha permitido comprobar que no existe un único modelo mejor en todas las métricas, sino que el rendimiento depende del criterio de evaluación considerado.

El modelo **HistGradientBoosting** ha sido el más interesante desde el punto de vista de detección de impagos, ya que mejora claramente al modelo base en precision, recall y PR-AUC. Aunque su accuracy es inferior a la del baseline, ofrece un comportamiento más útil desde el punto de vista del riesgo, porque identifica una proporción mucho mayor de clientes problemáticos.

La **SVM** no ha conseguido mejorar el rendimiento del modelo base, lo que puede deberse a que este tipo de modelo es sensible a la elección de hiperparámetros, al escalado de las variables y al tamaño de la muestra utilizada en el entrenamiento. En cuanto al **MLP**, aunque ha alcanzado la mayor accuracy y una precision elevada, su recall ha sido demasiado bajo, lo que indica que tiende a clasificar la mayoría de los casos como no impago y, por tanto, no resulta adecuado si se busca detectar de forma efectiva la clase minoritaria.

En conclusión, el pipeline propuesto sí aporta mejoras respecto al modelo base, pero dichas mejoras dependen de la métrica considerada. Si lo más importante es detectar impagos, el Random Forest es la alternativa más adecuada entre las probadas. Estos resultados también ponen de manifiesto la importancia de evaluar modelos desbalanceados con métricas más allá de la accuracy, ya que una accuracy alta no garantiza una buena detección de la clase de interés.